<a href="https://colab.research.google.com/github/bimbache62-ctrl/TFM/blob/main/C%C3%B3digos_TFM_M%C3%B3dulo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



**TRABAJO DE FIN DE MÁSTER (TFM)**


**ANEXO METODOLÓGICO: PIPELINE BIOINFORMÁTICO (MODELO V31)**
Este script centraliza el post-procesado de los archivos de antiSMASH, la
reducción de redundancia mediante CD-HIT, la auditoría de secuencias, la
transformación algebraica TF-IDF y el entramado predictivo de Machine Learning
(Isolation Forest, XGBoost y SHAP).

Se ha estructurado para ejecutarse de forma secuencial garantizando la
reproducibilidad de los resultados presentados en la memoria.



📂** Estructura del Repositorio y Correspondencia con la Memoria**
El código se ha dividido lógicamente en los siguientes módulos, correspondientes a los Anexos detallados en el apartado de Materiales y Métodos (3.4.3 y 3.5):

01_extraccion_antismash.py (Anexo 3.7.1 - Post-procesado de antiSMASH)

02_clustering_y_hmmer.py (Anexo 3.7.1 - CD-HIT y API EBI)

03_matriz_tfidf.py (Anexo 3.7.2 - Construcción matricial)

04_modelo_v31_xgboost.py (Anexo 3.7.4 - Machine Learning y SHAP)

 **Módulo 1: Extracción, Tipado Automatizado y Minería Funcional de Clústeres Biosintéticos (BGCs)**

Este módulo inicial gestiona la ingesta masiva de datos genómicos y la extracción sistemática de los Clústeres de Genes Biosintéticos (BGCs) a partir de los outputs crudos generados por antiSMASH.

Procesamiento e Ingesta: Automatiza la descompresión de archivos genómicos (.zip) estructurados por aislamiento/cepa fúngica y el parseo de registros secuenciales en formato GenBank (.gbk).

Corrección de Clústeres Híbridos: Implementa un control de calidad que detecta y corrige anomalías sintácticas en la clasificación de productos biosintéticos híbridos (p. ej., fusiones NRPS-PKS), reestructurando su nomenclatura para evitar sesgos numéricos posteriores en las matrices pangenómicas.

Clasificación Ontológica Estricta: Evalúa la ontología génica de cada locus mediante un filtro heurístico riguroso basado en palabras clave específicas y perfiles Pfam. Segmenta los elementos de la región en cinco categorías funcionales discretas: Core (genes biosintéticos clave), Accessory (enzimas adicionales de modificación), Transporter (bombas de eflujo y permeasas ABC/MFS), Regulator (factores de transcripción y proteínas de unión a ADN) y Other.

Anotación Remota por HMMER/Pfam: En caso de que un gen biosintético carezca de dominios preestablecidos, el pipeline integra consultas asíncronas a la API REST de HMMER (EBI) mediante el servicio hmmscan contra la base de datos Pfam, blindando la completitud de la caracterización funcional.

Minería de Similitud MIBiG: Ejecuta una minería directa sobre los archivos de alineamiento knownclusterblast.txt para extraer de forma inequívoca el identificador del BGC homólogo en el repositorio MIBiG junto con su porcentaje exacto de identidad evolutiva.

In [ ]:
# =====================================================================
# INSTALACIÓN DE DEPENDENCIAS Y SOFTWARE EXTERNO
# =====================================================================

# 1. Librerías de Python (Secuencias, Gráficos, Clinker y Excel)
!pip install biopython upsetplot dna_features_viewer clinker openpyxl

# 2. Software de línea de comandos (CD-HIT para clustering)
!apt-get update
!apt-get install -y cd-hit
# 3. Para machine learning
!pip install xgboost shap scikit-learn biopython openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,959 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 http://security.ubuntu.

In [ ]:
import os
import glob
import zipfile
import shutil
import re
import warnings
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import requests
import time
from typing import Tuple
from Bio import SeqIO, BiopythonWarning
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

# Dependencias opcionales para gráficas
try:
    from upsetplot import plot as upset_plot
    USE_UPSET = True
except ImportError:
    USE_UPSET = False

try:
    from dna_features_viewer import GraphicFeature, GraphicRecord
    USE_DFV = True
except ImportError:
    USE_DFV = False

# =====================================================================
# Configuración inicial y rutas
# =====================================================================
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=BiopythonWarning)
pd.set_option('future.no_silent_downcasting', True)

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PATH_DRIVE = '/content/drive/MyDrive/TFM/antismash'
PATH_WORK = '/content/datos_botrytis'
PATH_CLINKER = '/content/agrupaciones_clinker'
PATH_RESULTS = os.path.join(PATH_DRIVE, "Resultados_Pangenomicos_Final")

for d in [PATH_WORK, PATH_CLINKER, PATH_RESULTS]:
    os.makedirs(d, exist_ok=True)

# =====================================================================
# Funciones auxiliares (HMMER, MIBiG, Ontología, Gráficos)
# =====================================================================
def buscar_pfam_online(secuencia: str) -> str:
    """Consulta la API de HMMER (EBI) mediante secuencia FASTA."""
    if not secuencia or len(secuencia) < 15:
        return "Invalid_Sequence"

    url = 'https://www.ebi.ac.uk/Tools/hmmer/search/hmmscan'
    headers = {'Accept': 'application/json'}

    data = {
        'hmmdb': 'pfam',
        'seq': f">query_sequence\n{secuencia}"
    }

    try:
        time.sleep(1.5) # Evitar saturar la API
        response = requests.post(url, headers=headers, data=data, timeout=45)

        if response.status_code == 200:
            json_data = response.json()
            pfams = set()
            hits = json_data.get('results', {}).get('hits', [])
            for hit in hits:
                acc = hit.get('acc', '')
                if acc.startswith('PF'):
                    pfams.add(acc.split('.')[0])

            return " | ".join(sorted(list(pfams))) if pfams else "No_Pfam_Hits"
        else:
            print(f"  [!] Error en API HMMER: {response.status_code}")
            return "Not_Annotatable_Online"

    except Exception:
        return "Network_Timeout"

def minar_raw_mibig(gbk_path: str) -> Tuple[str, int]:
    """Extrae el hit de MIBiG y la similitud del archivo txt generado por antiSMASH."""
    dir_base = os.path.dirname(gbk_path)
    txt_files = glob.glob(os.path.join(dir_base, "**", "*knownclusterblast*.txt"), recursive=True)

    for txt_path in txt_files:
        try:
            with open(txt_path, 'r') as f:
                for line in f:
                    if "BGC0" in line:
                        data = line.strip().split('\t')
                        bgc_id = next((x for x in data if x.startswith("BGC")), "Orphan")
                        nums = re.findall(r'\d+', line)
                        sim = int(nums[-1]) if nums else 0
                        return bgc_id, sim
        except Exception: continue
    return "Orphan", 0

def clasificar_gen_ontologia_estricta(gen_txt: str, gene_kind: str) -> str:
    """Clasifica la función del gen basándose en keywords y pfams."""
    gen_txt_lower = gen_txt.lower()

    is_core_domain = any(k in gen_txt_lower for k in [
        'nrps', 'non-ribosomal peptide', 'polyketide synthase', 'pks',
        'terpene synthase', 'terpene cyclase', 'phytoene synthase', 'prenyltransferase',
        'sesquiterpene', 'diterpene', 'cyclase'
    ])
    if gene_kind == 'biosynthetic' or is_core_domain: return 'Core'
    if any(k in gen_txt_lower for k in ['transport', 'efflux', 'permease', 'pump', 'abc', 'mfs', 'pf00083', 'pf07690', 'pf00664']): return 'Transporter'
    if any(k in gen_txt_lower for k in ['regulat', 'transcription', 'binding', 'pf00196', 'pf04083', 'pf00228']): return 'Regulator'

    accesorios_keywords = ['p450', 'cytochrome', 'pf00067', 'dehydrogenase', 'reductase', 'oxidase', 'methyltransferase', 'halogenase', 'mutase', 'isomerase']
    if gene_kind == 'biosynthetic-additional' or any(k in gen_txt_lower for k in accesorios_keywords): return 'Accessory'
    return 'Other'

def dibujar_singleton_funcional(gbk_path: str, output_svg: str):
    """Mapeo del cluster lineal coloreado por función (SVG)."""
    if not USE_DFV: return
    try:
        record = SeqIO.read(gbk_path, "genbank")
        features_to_plot = []
        for feat in record.features:
            if feat.type == "CDS":
                gen_txt = str(feat.qualifiers)
                clase_gen = clasificar_gen_ontologia_estricta(gen_txt, feat.qualifiers.get('gene_kind', [''])[0].lower())

                # Colores por categoría funcional
                if clase_gen == 'Core': col = "#E74C3C"
                elif clase_gen == 'Accessory': col = "#2ECC71"
                elif clase_gen == 'Transporter': col = "#3498DB"
                elif clase_gen == 'Regulator': col = "#F1C40F"
                else: col = "#BDC3C7"

                label = feat.qualifiers.get('gene', feat.qualifiers.get('locus_tag', ['']))[0]
                features_to_plot.append(GraphicFeature(start=int(feat.location.start),
                                                       end=int(feat.location.end),
                                                       strand=feat.location.strand,
                                                       color=col, label=label))

        graf_record = GraphicRecord(sequence_length=len(record), features=features_to_plot)
        fig, ax = plt.subplots(figsize=(12, 3))
        graf_record.plot(ax=ax)
        plt.title(f"Functional Architecture (Singleton): {os.path.basename(gbk_path)}", pad=20)
        fig.savefig(output_svg, bbox_inches='tight', format='svg')
        plt.close(fig)
    except Exception as e:
        print(f"  [!] No se pudo dibujar el Singleton {gbk_path}: {e}")

# =====================================================================
# Pipeline principal
# =====================================================================

print("Paso 1: Extrayendo genomas y parseando BGCs...")
for zip_file in glob.glob(os.path.join(PATH_DRIVE, '*.zip')):
    sp_name = os.path.basename(zip_file).replace('.zip', '')
    sp_dir = os.path.join(PATH_WORK, sp_name)
    os.makedirs(sp_dir, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(sp_dir)

datos_bgc = []
fasta_records_cdhit = []
dict_protein_sequences = {}

especies_extraidas = [d for d in os.listdir(PATH_WORK) if os.path.isdir(os.path.join(PATH_WORK, d))]

for nombre_especie in especies_extraidas:
    ruta_especie = os.path.join(PATH_WORK, nombre_especie)

    for gbk in glob.glob(os.path.join(ruta_especie, "**", "*region*.gbk"), recursive=True):
        try:
            record = SeqIO.read(gbk, "genbank")
            mibig_id, similitud = minar_raw_mibig(gbk)

            # Extraer producto y manejar clusters híbridos
            raw_product = "Unknown"
            for feat in record.features:
                if feat.type in ['region', 'cluster']:
                    productos = feat.qualifiers.get('product', [raw_product])
                    raw_product = "-".join(productos)
                    break

            clean_product = raw_product.replace('-like', '_like').replace(' ', '')
            if '-' in clean_product or ',' in clean_product:
                categoria_cluster, subtipo = "Hybrid", raw_product
            else:
                categoria_cluster, subtipo = raw_product.capitalize(), raw_product

            counts = {"Total_Genes": 0, "Core_Genes": 0, "Accessory_Genes": 0, "Transporters": 0, "Regulators": 0, "Others": 0}
            dominios_set = set()
            dominios_core_set = set()
            prot_sec_core, prot_sec_fallback = "", ""

            # Conteo de genes y extracción de dominios
            for feat in record.features:
                if feat.type == "CDS":
                    counts["Total_Genes"] += 1
                    gen_txt = str(feat.qualifiers)
                    gene_kind = feat.qualifiers.get('gene_kind', [''])[0].lower()

                    dominios_gen = []
                    if 'sec_met_domain' in feat.qualifiers:
                        dominios_gen.extend(feat.qualifiers['sec_met_domain'])
                    elif 'gene_functions' in feat.qualifiers:
                        for gf in feat.qualifiers['gene_functions']:
                            clean_gf = gf.replace("biosynthetic-additional (smcogs) ", "").replace("other (smcogs) ", "")
                            dominios_gen.append(clean_gf)
                    elif 'NRPS_PKS' in feat.qualifiers:
                        dominios_gen.extend([d for d in feat.qualifiers['NRPS_PKS'] if "Domain:" in d])

                    if not dominios_gen:
                        raw_pfams = re.findall(r'PF\d{5}', gen_txt, re.IGNORECASE) + re.findall(r'PFAM\d{5}', gen_txt, re.IGNORECASE)
                        dominios_gen.extend([p.upper().replace('PFAM', 'PF') for p in raw_pfams])

                    dominios_gen = [" ".join(d.split()) for d in dominios_gen]
                    dominios_set.update(dominios_gen)

                    traduccion = feat.qualifiers.get('translation', [''])[0]
                    if len(traduccion) > len(prot_sec_fallback): prot_sec_fallback = traduccion

                    clase_gen = clasificar_gen_ontologia_estricta(gen_txt, gene_kind)

                    if clase_gen == 'Core':
                        counts["Core_Genes"] += 1
                        dominios_core_set.update(dominios_gen)
                        if len(traduccion) > len(prot_sec_core): prot_sec_core = traduccion
                    elif clase_gen == 'Accessory': counts["Accessory_Genes"] += 1
                    elif clase_gen == 'Transporter': counts["Transporters"] += 1
                    elif clase_gen == 'Regulator': counts["Regulators"] += 1
                    else: counts["Others"] += 1

                    if traduccion:
                        prot_id = f"{nombre_especie}_{os.path.basename(gbk)}_{feat.qualifiers.get('locus_tag', ['ND'])[0]}"
                        dict_protein_sequences[prot_id] = SeqRecord(Seq(traduccion), id=prot_id, description=f"[{clase_gen}]")

            # Normalizar conteo de core genes si hay más de 1 en no-híbridos
            if categoria_cluster != "Hybrid" and counts["Core_Genes"] > 1:
                exceso_core = counts["Core_Genes"] - 1
                counts["Core_Genes"] = 1
                counts["Accessory_Genes"] += exceso_core

            bgc_id = f"{nombre_especie}_{os.path.basename(gbk)}"
            secuencia_final = prot_sec_core if prot_sec_core else prot_sec_fallback

            if secuencia_final:
                fasta_records_cdhit.append(SeqRecord(Seq(secuencia_final), id=bgc_id, description=""))

            if dominios_core_set:
                dominios_core_final = " | ".join(sorted(list(dominios_core_set)))
            elif secuencia_final:
                print(f"  -> Buscando dominios core online para {bgc_id}...")
                dominios_core_final = buscar_pfam_online(secuencia_final)
                if dominios_core_final not in ["No_Pfam_Hits", "Invalid_Sequence", "Network_Timeout", "Not_Annotatable_Online"]:
                    dominios_set.update(dominios_core_final.split(" | "))
            else:
                dominios_core_final = "Not_Annotated_No_Sequence"

            datos_bgc.append({
                "Species": nombre_especie, "BGC_ID": bgc_id, "Category": categoria_cluster,
                "Original_Subtype": subtipo, "MIBiG_Hit": mibig_id, "MIBiG_Identity_%": similitud,
                "Total_Identified_Domains": " | ".join(sorted(list(dominios_set))) if dominios_set else "None",
                "Core_Identified_Domains": dominios_core_final,
                "Core_Sequence": secuencia_final,
                **counts
            })

            # Preparar GBK para Clinker (limpiar metadata para evitar nombres duplicados en los plots)
            record.id = bgc_id
            record.name = bgc_id[:16]
            record.description = ""
            record.annotations.pop('organism', None)
            record.annotations.pop('source', None)
            SeqIO.write(record, os.path.join(PATH_CLINKER, f"{bgc_id}.gbk"), "genbank")

        except Exception: pass

fasta_path_global = "/content/cores_bgcs.fasta"
SeqIO.write(fasta_records_cdhit, fasta_path_global, "fasta")

print("\nPaso 2: Ejecutando CD-HIT para generar familias homólogas...")
subprocess.run(f"cd-hit -i {fasta_path_global} -o /content/bgcs_clustered -c 0.7 -n 4 -d 0 -M 0 -G 0 -aS 0.6", shell=True, check=True, capture_output=True)

familias_reales = {}
with open("/content/bgcs_clustered.clstr", "r") as f:
    cluster_actual = ""
    for linea in f:
        if linea.startswith(">"): cluster_actual = f"Fam_{linea.strip().replace('>', '').replace(' ', '_')}"
        else:
            bgc_id = linea.split(">")[1].split("...")[0]
            familias_reales[bgc_id] = cluster_actual

df_master = pd.DataFrame(datos_bgc)
df_master["Homology_Family"] = df_master["BGC_ID"].map(familias_reales).fillna("Orphan")

for idx, row in df_master.iterrows():
    carpeta_clinker = os.path.join(PATH_CLINKER, row["Homology_Family"])
    os.makedirs(carpeta_clinker, exist_ok=True)
    ruta_gbk = os.path.join(PATH_CLINKER, f"{row['BGC_ID']}.gbk")
    if os.path.exists(ruta_gbk):
        shutil.move(ruta_gbk, os.path.join(carpeta_clinker, f"{row['BGC_ID']}.gbk"))

todas_las_especies = df_master['Species'].unique()
total_sp = len(todas_las_especies)

print("\nPaso 3: Analizando conservación y generando gráficos por categoría...")

df_resumen_clases = pd.crosstab(df_master['Species'], df_master['Category'])
df_resumen_clases.to_excel(os.path.join(PATH_RESULTS, "0_Resumen_Clases_por_Especie.xlsx"))

plt.figure(figsize=(12, 8))
df_resumen_clases.plot(kind='bar', stacked=True, colormap='tab20', ax=plt.gca())
plt.title("Biosynthetic Capacity by Species", fontsize=16, pad=20)
plt.ylabel("Number of Identified BGCs", fontsize=12)
plt.xlabel("Isolate / Species", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title="Metabolite Type", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(PATH_RESULTS, "0_Figura_Composicion_General.svg"), format='svg')
plt.close()

for tipo in df_master['Category'].unique():
    print(f"  Procesando: {tipo}")
    ruta_tipo = os.path.join(PATH_RESULTS, f"Category_{tipo}")
    os.makedirs(ruta_tipo, exist_ok=True)

    df_tipo = df_master[df_master['Category'] == tipo]

    cols = ["Species", "BGC_ID", "Category", "Original_Subtype", "MIBiG_Hit", "MIBiG_Identity_%",
            "Total_Genes", "Core_Genes", "Accessory_Genes", "Transporters", "Regulators",
            "Others", "Total_Identified_Domains", "Core_Identified_Domains", "Homology_Family"]
    df_tipo[cols].to_excel(os.path.join(ruta_tipo, f"1_Anatomy_{tipo}.xlsx"), index=False)

    matriz_pa = pd.crosstab(df_tipo['Species'], df_tipo['Homology_Family'])
    matriz_pa = matriz_pa.reindex(todas_las_especies, fill_value=0)
    matriz_pa.to_excel(os.path.join(ruta_tipo, f"2_PA_Matrix_{tipo}.xlsx"))

    stats = []
    path_core = os.path.join(ruta_tipo, "Synteny_STRICT_CORE")
    path_sing = os.path.join(ruta_tipo, "Synteny_EXCLUSIVE_SINGLETON")

    for familia in matriz_pa.columns:
        n_sp = (matriz_pa[familia] > 0).sum()
        ruta_origen = os.path.join(PATH_CLINKER, familia)

        if n_sp == total_sp:
            estado = "Strict Core"
            os.makedirs(path_core, exist_ok=True)
            # Sintenia interactiva con Clinker
            archivo_salida = os.path.join(path_core, f"Synteny_CORE_{familia}.html")
            os.system(f"clinker {ruta_origen}/*.gbk -p {archivo_salida} -i 0.4")

        elif n_sp == 1:
            estado = "Specific Singleton"
            os.makedirs(path_sing, exist_ok=True)

            datos_fam = df_tipo[df_tipo['Homology_Family'] == familia].iloc[0]
            gbk_singleton = os.path.join(ruta_origen, f"{datos_fam['BGC_ID']}.gbk")
            archivo_salida_svg = os.path.join(path_sing, f"Architecture_SINGLETON_{familia}.svg")

            dibujar_singleton_funcional(gbk_singleton, archivo_salida_svg)

        else:
            estado = f"Accessory ({n_sp}/{total_sp})"

        stats.append({"Homology_Family": familia, "Prevalence": f"{n_sp}/{total_sp}", "Evolutionary_State": estado})

    pd.DataFrame(stats).to_excel(os.path.join(ruta_tipo, f"3_Conservation_{tipo}.xlsx"), index=False)

    if USE_UPSET:
        if len(matriz_pa.index) > 1 and len(matriz_pa.columns) > 0:
            try:
                df_upset_bool = (matriz_pa > 0).astype(bool).T
                df_upset_counts = df_upset_bool.groupby(list(df_upset_bool.columns)).size()

                plt.figure(figsize=(10, 6))
                upset_plot(df_upset_counts, sort_by='cardinality', show_counts=True)
                plt.title(f"BGC Pangenomic Intersection - Category: {tipo}", pad=20)
                plt.savefig(os.path.join(ruta_tipo, f"4_UpSet_Plot_{tipo}.svg"), format='svg', bbox_inches='tight')
                plt.close()
            except Exception as e:
                print(f"  [!] Fallo al generar UpSet plot para {tipo}: {e}")
                plt.close()

print("\nPaso 4: Exportando datos para modelos predictivos...")
path_ml_excel = os.path.join(PATH_RESULTS, "Data_Machine_Learning_ML.xlsx")
path_ml_pkl = os.path.join(PATH_RESULTS, "Data_Machine_Learning_ML.pkl")

df_master.to_excel(path_ml_excel, index=False)
df_master.to_pickle(path_ml_pkl)
print(f"  -> Archivos guardados: .xlsx y .pkl")

print("\nAnálisis finalizado. Los resultados se han guardado en el directorio de salida.")

Mounted at /content/drive
Paso 1: Extrayendo genomas y parseando BGCs...

Paso 2: Ejecutando CD-HIT para generar familias homólogas...

Paso 3: Analizando conservación y generando gráficos por categoría...
  Procesando: Nrps-like
  Procesando: T3pks
  Procesando: T1pks
  Procesando: Nrps
  Procesando: Terpene
  Procesando: Hybrid
  Procesando: Other
  Procesando: Indole
  Procesando: Napaa

Paso 4: Exportando datos para modelos predictivos...
  -> Archivos guardados: .xlsx y .pkl

Análisis finalizado. Los resultados se han guardado en el directorio de salida.


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>